### Dependency Installation

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install  "trl<0.9.0" bitsandbytes accelerate peft

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments, TextStreamer
from datasets import load_dataset

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3-8b-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

# Ensure pad token exists (causal LMs sometimes lack it)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    if hasattr(model, "config"):
        model.config.pad_token_id = tokenizer.pad_token_id


## Dataset prep

In [ ]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

dataset = load_dataset("unsloth/alpaca-cleaned", split="train")
dataset = dataset.select(range(500))

EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]

    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        if inp and len(str(inp).strip()) > 0:
            text = alpaca_prompt.format(instruction, inp, output) + EOS_TOKEN
        else:
            text = alpaca_prompt.format(instruction, "", output) + EOS_TOKEN
        texts.append(text)

    # IMPORTANT: return a dict of lists for batched=True
    return {"text": texts}

# IMPORTANT: map must be OUTSIDE the function and not indented under return
dataset = dataset.map(formatting_prompts_func, batched=True, desc="Formatting to text")

In [ ]:
print(dataset.column_names)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # 8/16/32 are common
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,            # 0 is optimal in unsloth’s kernels
    bias="none",
    use_gradient_checkpointing="unsloth",  # True or "unsloth" for long context
    random_state=3407,
)


In [ ]:
training_arguments = TrainingArguments(
    output_dir="./outputs",               # safer than absolute /outputs
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    optim="adamw_8bit",                   # requires bitsandbytes + compatible GPU
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,                # now contains a 'text' column
    dataset_text_field="text",
    max_seq_length=2048,
    args=training_arguments,
    # packing=False is default; keep it off for Alpaca-style text rows
)

trainer_stats = trainer.train()

<a name="Inference"></a>
### Inference
Let's run the model

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonacci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Continue the fibonacci sequence.", # instruction
        "1, 1, 2, 3, 5, 8", # input
        "", # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)